<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/exercises/ex4/ex4_ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML4NLP1
## Starting Point for Exercise 4

This notebook is the starting point for exercise 4. It has two parts:

1. **Part 1 (BERT):** fine-tune a pretrained BERT model for named entity recognition (NER, see [Section 18.3 in Jurafsky & Martin](https://web.stanford.edu/~jurafsky/slp3/18.pdf)) in a language of your choice, in four variants.
2. **Part 2 (GLiNER):** use GLiNER for zero-shot NER, compare it with your fine-tuned models on the same test set, and use it to anonymise your own texts.

Part 1 needs a GPU (Google Colab or JupyterHub); Part 2 also runs on a CPU.

Throughout the notebook, there are questions that you should address in your report. These are marked with 📝❓ and are collected again in the lab report section at the end of the notebook.

## Use of generative AI

AI tools may be used in this exercise to help you explain code, debug, improve documentation, and explore alternatives. Any use must be documented in the *Declaration on the use of AI* at the end of this notebook (which tools were used and for which purposes). You remain responsible for understanding, checking, and testing everything you submit, and you must be able to explain any submitted text, code, analyses, and AI-assisted contributions. See the [CL guidelines on academic integrity and AI use](https://www.cl.uzh.ch/en/Studies/Services/academic-integrity.html).

# Setup

In [1]:
# Pin the library versions used in this exercise,
# so that your notebook runs the same way for you and for your reviewers.
%pip install -q transformers==5.16.1 datasets==5.1.0 accelerate==1.15.0 gliner==0.2.29 seqeval==1.2.2 sentencepiece protobuf

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='Glyph .* missing from font')
warnings.filterwarnings('ignore', message='.*IProgress not found.*')
warnings.filterwarnings('ignore', message='.*resume_download.*')
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

import torch
from transformers import set_seed

# Pick the best available device: NVIDIA GPU, then Apple Silicon GPU, then CPU
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(device)

# Fix the random seed, so that rerunning the notebook gives the same results on the same hardware
set_seed(42)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
cuda


# Part 1: Named entity recognition with BERT

## Load and prepare the required data

In [2]:
# Choose a supported language, apart from English (see the exercise description for the conditions).
# Examples: "de", "fr", "es", "it".
# NOTE: See dataset card for supported languages (https://huggingface.co/datasets/unimelb-nlp/wikiann)
chosen_language_code = "de"

In [3]:
import datasets

# NOTE: If the maximum sequence length exceeds the model's maximum
# sequence length, you need to make adjustments (for example, when
# choosing 'en')
test_set = datasets.load_dataset("unimelb-nlp/wikiann", chosen_language_code, split="test[:2000]")
train_set1000 = datasets.load_dataset("unimelb-nlp/wikiann", chosen_language_code, split="train[:1000]")
train_set3000 = datasets.load_dataset("unimelb-nlp/wikiann", chosen_language_code, split="train[:3000]")

README.md:   0%|          | 0.00/158k [00:00<?, ?B/s]

de/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  835kB            

de/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

de/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  832kB            

de/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

de/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.67MB            

de/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

**NOTE: Make sure that there are indeed as many data points in the above sets**

In [4]:
print(train_set1000)
print(train_set3000)
print(test_set)

Dataset({
    features: ['tokens', 'ner_tags', 'langs', 'spans'],
    num_rows: 1000
})
Dataset({
    features: ['tokens', 'ner_tags', 'langs', 'spans'],
    num_rows: 3000
})
Dataset({
    features: ['tokens', 'ner_tags', 'langs', 'spans'],
    num_rows: 2000
})


### Named entity recognition and the IOB format

In NER, the model assigns a tag to every word of a sentence. WikiANN has three entity types: persons (`PER`), organisations (`ORG`) and locations (`LOC`). Since an entity can span several words, the tags follow the IOB format (also called BIO): `B-` marks the first word of an entity, `I-` marks every following word of the same entity, and `O` marks words outside any entity. For example:

| Angela | Merkel | visited | the | University | of | Zurich |
|---|---|---|---|---|---|---|
| B-PER | I-PER | O | O | B-ORG | I-ORG | I-ORG |

📝❓ Why do we need separate `B-` and `I-` tags? What could not be represented with only `PER`, `ORG`, `LOC` and `O`?

In [5]:
# WikiANN stores the tags as integers. This dictionary maps each tag name to its integer;
# you will need it to set the number of labels of your model and to convert
# the predicted integers back to tag names for the evaluation.
ner_tags = {tag: i for i, tag in enumerate(test_set.features["ner_tags"].feature.names)}
print(ner_tags)

{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6}


**TODO: Inspect and Describe the Data, including Average and Maximum Input length (in tokens)**

📝❓Why do you need to be aware of the longest input length within your dataset? Which parameter of the model dictates this?

In [6]:
from transformers import AutoTokenizer
import torch

# TODO: Load the tokenizer
model_name = "google-bert/bert-base-german-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/255k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/485k [00:00<?, ?B/s]

In [7]:
# TODO: Adjust by actually finding the maximum sequence length
from collections import Counter
import numpy as np

id2tag = {i: tag for tag, i in ner_tags.items()}

# Inspect an example sample from the dataset
print("Example tokens:", train_set1000[0]["tokens"])
print("Example NER tags:", [id2tag[t] for t in train_set1000[0]["ner_tags"]])
print("-" * 60)

# Inspect NER tag distribution and share of 'O' tags in the test set
test_tag_counts = Counter(tag for tags in test_set["ner_tags"] for tag in tags)
total_test_words = sum(test_tag_counts.values())
print("Test set NER tag distribution:")
for tag_name, tag_id in ner_tags.items():
    count = test_tag_counts[tag_id]
    print(f"  {tag_name:6s} (id={tag_id}): {count:6d} ({count / total_test_words:.2%})")
print("-" * 60)

# Compute average and maximum lengths in words and subword tokens across splits
all_max_token_lengths = []
for split_name, ds in [("train_set1000", train_set1000), ("train_set3000", train_set3000), ("test_set", test_set)]:
    word_lengths = [len(tokens) for tokens in ds["tokens"]]
    tokenized_lengths = [
        len(tokenizer(tokens, is_split_into_words=True, truncation=False)["input_ids"])
        for tokens in ds["tokens"]
    ]
    all_max_token_lengths.append(max(tokenized_lengths))
    print(
        f"{split_name:14s} | Words: avg={np.mean(word_lengths):.2f}, max={max(word_lengths):3d} "
        f"| Subword tokens (incl. [CLS]/[SEP]): avg={np.mean(tokenized_lengths):.2f}, max={max(tokenized_lengths):3d}"
    )

max_sequence_length = min(max(all_max_token_lengths), tokenizer.model_max_length)

Example tokens: ['als', 'Teil', 'der', 'Savoyer', 'Voralpen', 'im', 'Osten', '.']
Example NER tags: ['O', 'O', 'O', 'B-LOC', 'I-LOC', 'O', 'O', 'O']
------------------------------------------------------------
Test set NER tag distribution:
  O      (id=0):  13458 (69.86%)
  B-PER  (id=1):    938 (4.87%)
  I-PER  (id=2):   1322 (6.86%)
  B-ORG  (id=3):    823 (4.27%)
  I-ORG  (id=4):   1270 (6.59%)
  B-LOC  (id=5):    973 (5.05%)
  I-LOC  (id=6):    481 (2.50%)
------------------------------------------------------------
train_set1000  | Words: avg=9.77, max= 76 | Subword tokens (incl. [CLS]/[SEP]): avg=16.41, max=114
train_set3000  | Words: avg=9.74, max= 76 | Subword tokens (incl. [CLS]/[SEP]): avg=16.30, max=114
test_set       | Words: avg=9.63, max= 45 | Subword tokens (incl. [CLS]/[SEP]): avg=16.04, max= 63


In [8]:
print(max_sequence_length)

114


📝❓The dataset is split into words, and the assigned labels are for words. How should we deal with labels **after** tokenization? NOTE: Each word may be split into one or multiple tokens by the tokenizer.

In [9]:
# TODO: Implement this function
def encode_and_align_labels(dataset, tokenizer, max_sequence_length):
    """Tokenizes the input tokens and aligns the word-level NER labels with the tokenized output."""
    def _tokenize_and_align(examples):
        tokenized_inputs = tokenizer(
            examples["tokens"],
            truncation=True,
            padding="max_length",
            max_length=max_sequence_length,
            is_split_into_words=True,
        )
        aligned_labels = []
        for i, word_labels in enumerate(examples["ner_tags"]):
            word_ids = tokenized_inputs.word_ids(batch_index=i)
            previous_word_idx = None
            label_ids = []
            for word_idx in word_ids:
                if word_idx is None:
                    # Special tokens ([CLS], [SEP], [PAD]) are ignored by cross-entropy loss
                    label_ids.append(-100)
                elif word_idx != previous_word_idx:
                    # Assign the word's NER tag to its first subword token
                    label_ids.append(word_labels[word_idx])
                else:
                    # Ignore subsequent subword tokens of the same word
                    label_ids.append(-100)
                previous_word_idx = word_idx
            aligned_labels.append(label_ids)

        tokenized_inputs["labels"] = aligned_labels
        return tokenized_inputs

    return dataset.map(_tokenize_and_align, batched=True)

In [10]:
# TODO: Encode the two training sets and the test set by applying the function above
encoded_test_set = encode_and_align_labels(test_set, tokenizer, max_sequence_length)
encoded_train_set1000 = encode_and_align_labels(train_set1000, tokenizer, max_sequence_length)
encoded_train_set3000 = encode_and_align_labels(train_set3000, tokenizer, max_sequence_length)

# Set format for PyTorch
encoded_test_set.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)
encoded_train_set1000.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)
encoded_train_set3000.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [11]:
# Check out how the training sets are encoded
for key, val in encoded_train_set1000[0].items():
    print(f'{key}: {val.size()}')

input_ids: torch.Size([114])
attention_mask: torch.Size([114])
labels: torch.Size([114])


Example of how your output could look like.

input_ids: torch.Size([???])

attention_mask: torch.Size([???])

labels: torch.Size([???])

📝❓What value should replace the question marks in your print? Should this be the same for all samples? Why/Why not?

## Training

### Training utils

In [12]:
from transformers import AutoModelForTokenClassification, Trainer, TrainingArguments
import os
os.environ["WANDB_MODE"] = "disabled"

**TODO: Complete the following, reusable functions**

We evaluate on the level of entities, which is the standard for NER: the predicted tags of a sentence are first grouped into entities (for example, `B-PER I-PER` is one `PER` entity), and an entity only counts as correct if both its boundaries and its type match the gold entity. The [seqeval](https://github.com/chakki-works/seqeval) library does this for you. Note that `O` is not an entity type: it only marks words outside entities.

📝❓ Why do we evaluate NER on the level of entities rather than on the level of individual tokens? Hint: what score would a model get on the token level if it predicted `O` for every token? Check the share of `O` tags in your test set.

In [13]:
from seqeval.metrics import f1_score
import numpy as np

id2tag = {i: tag for tag, i in ner_tags.items()}


def compute_metrics(preds):
    """
    Compute entity-level macro and micro F1 scores from PredictionOutput

    seqeval expects one list of tag strings (for example "B-PER") per sentence,
    for the gold tags and for the predicted tags.

    Args:
        preds: transformers.trainer_utils.PredictionOutput

    Returns:
        dict with macro_f1 and micro_f1 scores
    """
    predictions = np.argmax(preds.predictions, axis=-1)
    labels = preds.label_ids

    true_tags = [
        [id2tag[int(label_id)] for pred_id, label_id in zip(pred_seq, label_seq) if label_id != -100]
        for pred_seq, label_seq in zip(predictions, labels)
    ]
    pred_tags = [
        [id2tag[int(pred_id)] for pred_id, label_id in zip(pred_seq, label_seq) if label_id != -100]
        for pred_seq, label_seq in zip(predictions, labels)
    ]

    return {
        "micro_f1": float(f1_score(true_tags, pred_tags, average="micro")),
        "macro_f1": float(f1_score(true_tags, pred_tags, average="macro")),
    }

In [14]:
def freeze_weights(model):
    """Freeze the weights for a given model.

    Args:
        model: transformers.PreTrainedModel

    Returns:
        model: transformers.PreTrainedModel
    """
    for param in model.base_model.parameters():
        param.requires_grad = False
    return model


def count_trainable_parameters(model):
    """Return the number of trainable parameters and total parameters."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

### Recommended hyperparameters

You do not need to tune the hyperparameters in this exercise. Please use the following settings, with a batch size of 16:

| Variants | Learning rate | Epochs |
|---|---|---|
| 1 and 2 (whole model fine-tuned) | 5e-5 | 3 |
| 3 and 4 (frozen backbone) | 1e-3 | 10 |

📝❓ The variants with a frozen backbone use a learning rate that is 20 times larger. Why do they need it? Think about which weights are trained in each case and where these weights start from (the load report that transformers prints when you load the model shows which weights are newly initialised).

## Variant 1: 1000 sentences, no frozen weights

**TODO: Initialise your model and set up your training arguments**

Note: if you pass the tokenizer to the `Trainer`, use the argument `processing_class` (the older argument `tokenizer` no longer exists in transformers 5).

In [15]:
import gc

variant_results = {}

model_v1 = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(ner_tags),
    id2label=id2tag,
    label2id=ner_tags,
)
trainable_v1, total_v1 = count_trainable_parameters(model_v1)
print(f"Variant 1 trainable parameters: {trainable_v1:,} / {total_v1:,}")

training_args_v1 = TrainingArguments(
    output_dir="./results/variant_1",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer_v1 = Trainer(
    model=model_v1,
    args=training_args_v1,
    train_dataset=encoded_train_set1000,
    eval_dataset=encoded_test_set,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  439MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google-bert/bert-base-german-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

Variant 1 trainable parameters: 108,496,135 / 108,496,135


**TODO: Train your model ⚡ GPU: about 3 min for variant 1 and up to 12 min for variant 4 on the course JupyterHub**

In [16]:
trainer_v1.train()

Step,Training Loss
63,0.497901
126,0.150341
189,0.074170


TrainOutput(global_step=189, training_loss=0.24080369333741525, metrics={'train_runtime': 63.9187, 'train_samples_per_second': 46.935, 'train_steps_per_second': 2.957, 'total_flos': 174545957916000.0, 'train_loss': 0.24080369333741525, 'epoch': 3.0})

**TODO: Compute Metrics/Performance of your model.**

📝❓ Is there a challenge when evaluating the predictions of your model? Why is this challenge present and how do you plan to deal with it?

Hint: Look at the lengths

To avoid rerunning, please also print the metrics of each model that completed training

In [17]:
preds_v1 = trainer_v1.predict(encoded_test_set)
metrics = compute_metrics(preds_v1)
variant_results["1. Whole model fine-tuned, 1000 sentences"] = {
    **metrics,
    "trainable_params": trainable_v1,
    "total_params": total_v1,
}
print(metrics)

{'micro_f1': 0.764367816091954, 'macro_f1': 0.76072558634563}


## Variant 2: 3000 sentences, no frozen weights

In [18]:
# Repeat after each run to save GPU memory (delete the previous model and trainer first)
del model_v1, trainer_v1
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()

model_v2 = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(ner_tags),
    id2label=id2tag,
    label2id=ner_tags,
)
trainable_v2, total_v2 = count_trainable_parameters(model_v2)
print(f"Variant 2 trainable parameters: {trainable_v2:,} / {total_v2:,}")

training_args_v2 = TrainingArguments(
    output_dir="./results/variant_2",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer_v2 = Trainer(
    model=model_v2,
    args=training_args_v2,
    train_dataset=encoded_train_set3000,
    eval_dataset=encoded_test_set,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer_v2.train()

preds_v2 = trainer_v2.predict(encoded_test_set)
metrics_v2 = compute_metrics(preds_v2)
variant_results["2. Whole model fine-tuned, 3000 sentences"] = {
    **metrics_v2,
    "trainable_params": trainable_v2,
    "total_params": total_v2,
}
print("Variant 2 Metrics:", metrics_v2)

# Token-level F1 (including 'O') for the Part 1 results
from sklearn.metrics import f1_score as sk_f1_score

v2_pred_ids = np.argmax(preds_v2.predictions, axis=-1)
v2_label_ids = preds_v2.label_ids
flat_true_v2 = [
    id2tag[int(l)] for pred_seq, label_seq in zip(v2_pred_ids, v2_label_ids)
    for p, l in zip(pred_seq, label_seq) if l != -100
]
flat_pred_v2 = [
    id2tag[int(p)] for pred_seq, label_seq in zip(v2_pred_ids, v2_label_ids)
    for p, l in zip(pred_seq, label_seq) if l != -100
]
token_micro_f1_v2 = float(sk_f1_score(flat_true_v2, flat_pred_v2, average="micro"))
token_macro_f1_v2 = float(sk_f1_score(flat_true_v2, flat_pred_v2, average="macro"))
print(f"Variant 2 Token-level (incl. 'O') -> micro_f1: {token_micro_f1_v2:.4f}, macro_f1: {token_macro_f1_v2:.4f}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google-bert/bert-base-german-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

Variant 2 trainable parameters: 108,496,135 / 108,496,135


Step,Training Loss
188,0.311847
376,0.108835
564,0.047673


Variant 2 Metrics: {'micro_f1': 0.8245741210583545, 'macro_f1': 0.8210286469932364}
Variant 2 Token-level (incl. 'O') -> micro_f1: 0.9527, macro_f1: 0.8805


## Variant 3: 1000 sentences, frozen weights

Note for Apple Silicon (MPS): training with a frozen backbone may fail with the error `scaled_dot_product_attention for MPS does not support dropout`. In that case, load the model with the additional argument `attn_implementation="eager"`.

In [19]:
del model_v2, trainer_v2
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()

extra_model_kwargs = {"attn_implementation": "eager"} if device == "mps" else {}
model_v3 = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(ner_tags),
    id2label=id2tag,
    label2id=ner_tags,
    **extra_model_kwargs,
)
model_v3 = freeze_weights(model_v3)
trainable_v3, total_v3 = count_trainable_parameters(model_v3)
print(f"Variant 3 trainable parameters: {trainable_v3:,} / {total_v3:,}")

training_args_v3 = TrainingArguments(
    output_dir="./results/variant_3",
    learning_rate=1e-3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer_v3 = Trainer(
    model=model_v3,
    args=training_args_v3,
    train_dataset=encoded_train_set1000,
    eval_dataset=encoded_test_set,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer_v3.train()

preds_v3 = trainer_v3.predict(encoded_test_set)
metrics_v3 = compute_metrics(preds_v3)
variant_results["3. Frozen backbone, 1000 sentences"] = {
    **metrics_v3,
    "trainable_params": trainable_v3,
    "total_params": total_v3,
}
print("Variant 3 Metrics:", metrics_v3)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google-bert/bert-base-german-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

Variant 3 trainable parameters: 5,383 / 108,496,135


Step,Training Loss
63,0.876017
126,0.449569
189,0.370921
252,0.344172
315,0.317485
378,0.306125
441,0.291746
504,0.291677
567,0.286271
630,0.281924


Variant 3 Metrics: {'micro_f1': 0.6225574220089133, 'macro_f1': 0.6159651626565749}


## Variant 4: 3000 sentences, frozen weights

In [20]:
del model_v3, trainer_v3
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()

extra_model_kwargs = {"attn_implementation": "eager"} if device == "mps" else {}
model_v4 = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(ner_tags),
    id2label=id2tag,
    label2id=ner_tags,
    **extra_model_kwargs,
)
model_v4 = freeze_weights(model_v4)
trainable_v4, total_v4 = count_trainable_parameters(model_v4)
print(f"Variant 4 trainable parameters: {trainable_v4:,} / {total_v4:,}")

training_args_v4 = TrainingArguments(
    output_dir="./results/variant_4",
    learning_rate=1e-3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer_v4 = Trainer(
    model=model_v4,
    args=training_args_v4,
    train_dataset=encoded_train_set3000,
    eval_dataset=encoded_test_set,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer_v4.train()

preds_v4 = trainer_v4.predict(encoded_test_set)
metrics_v4 = compute_metrics(preds_v4)
variant_results["4. Frozen backbone, 3000 sentences"] = {
    **metrics_v4,
    "trainable_params": trainable_v4,
    "total_params": total_v4,
}
print("Variant 4 Metrics:", metrics_v4)

# Clean up GPU memory after Variant 4
del model_v4, trainer_v4
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()

# Print Part 1 summary for all 4 variants
print("\n" + "=" * 75)
print("PART 1 SUMMARY: Entity-level F1 & Trainable Parameters")
print("=" * 75)
for name, res in variant_results.items():
    print(
        f"{name:42s} | micro_f1: {res['micro_f1']:.4f} | macro_f1: {res['macro_f1']:.4f} "
        f"| trainable params: {res['trainable_params']:,} / {res['total_params']:,}"
    )
print("-" * 75)
print(
    f"Variant 2 Comparison -> Entity-level: micro_f1={variant_results['2. Whole model fine-tuned, 3000 sentences']['micro_f1']:.4f}, "
    f"macro_f1={variant_results['2. Whole model fine-tuned, 3000 sentences']['macro_f1']:.4f} | "
    f"Token-level (incl. 'O'): micro_f1={token_micro_f1_v2:.4f}, macro_f1={token_macro_f1_v2:.4f}"
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google-bert/bert-base-german-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

Variant 4 trainable parameters: 5,383 / 108,496,135


Step,Training Loss
188,0.558442
376,0.331807
564,0.297463
752,0.287028
940,0.279021
1128,0.271170
1316,0.263128
1504,0.262578
1692,0.255242
1880,0.256524


Variant 4 Metrics: {'micro_f1': 0.6574358974358974, 'macro_f1': 0.6524071496803027}

PART 1 SUMMARY: Entity-level F1 & Trainable Parameters
1. Whole model fine-tuned, 1000 sentences  | micro_f1: 0.7644 | macro_f1: 0.7607 | trainable params: 108,496,135 / 108,496,135
2. Whole model fine-tuned, 3000 sentences  | micro_f1: 0.8246 | macro_f1: 0.8210 | trainable params: 108,496,135 / 108,496,135
3. Frozen backbone, 1000 sentences         | micro_f1: 0.6226 | macro_f1: 0.6160 | trainable params: 5,383 / 108,496,135
4. Frozen backbone, 3000 sentences         | micro_f1: 0.6574 | macro_f1: 0.6524 | trainable params: 5,383 / 108,496,135
---------------------------------------------------------------------------
Variant 2 Comparison -> Entity-level: micro_f1=0.8246, macro_f1=0.8210 | Token-level (incl. 'O'): micro_f1=0.9527, macro_f1=0.8805


## Part 1: Results

📝❓ Template:

Summary of the four model variants: entity-level micro and macro F1, and the number of trainable parameters

1. Whole model fine-tuned, 1000 sentences:
2. Whole model fine-tuned, 3000 sentences:
3. Frozen backbone, 1000 sentences:
4. Frozen backbone, 3000 sentences:

📝❓ For one of your variants, also compute the token-level micro and macro F1, including the `O` tag (for example, with `sklearn.metrics.f1_score` on the flattened word-level tags). Compare them with the entity-level micro F1 of the same variant. Why do these scores differ so much?

📝❓ Is it better to freeze or not to freeze the transformer backbone weights? Hypothesize why

# Part 2: Zero-shot NER and anonymisation with GLiNER

We will use a ***[fine-tuned version of GLiNER](https://huggingface.co/urchade/gliner_multi_pii-v1)***, specialized in personally identifiable information (PII).

GLiNER was trained on NER data that is mostly English and fine-tuned on personal data in English, French, German, Spanish, Portuguese and Italian. For other languages, it relies on its multilingual backbone, so its results may be noticeably lower. English entity labels are a safe choice, also for texts in other languages.

⚡ GOAL: Compare zero-shot NER with your fine-tuned models, then create your own small dataset with various PII mentions and anonymise it with GLiNER!

## Load the model

In [21]:
from gliner import GLiNER

# NOTE: No need to load the model on a GPU: all of Part 2 takes a few minutes on a CPU
gliner_model = GLiNER.from_pretrained("urchade/gliner_multi_pii-v1")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spm.model: downloading bytes:           |  0.00B            

In [22]:
# Code copy-pasted from the model card
text = """
Harilala Rasoanaivo, un homme d'affaires local d'Antananarivo, a enregistré une nouvelle société nommée "Rasoanaivo Enterprises" au Lot II M 92 Antohomadinika. Son numéro est le +261 32 22 345 67, et son adresse électronique est harilala.rasoanaivo@telma.mg. Il a fourni son numéro de sécu 501-02-1234 pour l'enregistrement.
"""

labels = ["work", "booking number", "personally identifiable information", "driver licence", "person", "book", "full address", "company", "actor", "character", "email", "passport number", "Social Security Number", "phone number"]
entities = gliner_model.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

Harilala Rasoanaivo => person
Rasoanaivo Enterprises => company
Lot II M 92 Antohomadinika => full address
+261 32 22 345 67 => phone number
harilala.rasoanaivo@telma.mg => email
501-02-1234 => Social Security Number


## Zero-shot NER on your test set

GLiNER has never been trained on WikiANN, and you do not train it either: you only give it the names of the entity types. Run it on the same 2,000 test sentences that you used in Part 1 and evaluate it in the same way, so that you can compare it with your four BERT variants.

GLiNER works on raw text and returns character spans, whereas WikiANN is split into words with one IOB tag per word. The helper function below joins the words of a sentence with spaces and converts the spans that GLiNER finds in this text back into IOB tags, so that you can evaluate them with seqeval as in Part 1.

If you run Part 2 in a new runtime (for example, on a CPU), first run the setup cell and the cells of Part 1 that load the data and define `ner_tags`.

In [23]:
# The entity types of WikiANN, as GLiNER labels
gliner_labels = {"person": "PER", "organization": "ORG", "location": "LOC"}


def spans_to_iob(words, entities):
    """Converts the entities that GLiNER finds in " ".join(words) into one IOB tag per word."""
    starts, position = [], 0
    for word in words:
        starts.append(position)
        position += len(word) + 1
    tags = ["O"] * len(words)
    for entity in sorted(entities, key=lambda e: e["start"]):
        inside = [i for i, start in enumerate(starts)
                  if start < entity["end"] and start + len(words[i]) > entity["start"]]
        for k, i in enumerate(inside):
            if tags[i] == "O":
                tags[i] = ("B-" if k == 0 else "I-") + gliner_labels[entity["label"]]
    return tags


# Example: the first test sentence
words = test_set[2]["tokens"]
entities = gliner_model.predict_entities(" ".join(words), list(gliner_labels), threshold=0.5)
print(words)
print(spans_to_iob(words, entities))

['Runde', 'rammte', 'Lorenzo', 'Bandini', 'an', 'zweiter', 'Stelle', 'liegend', 'die', 'Streckenbegrenzung', 'aus', 'Strohballen', '.']
['B-LOC', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'O']


**TODO: Run GLiNER on all test sentences and evaluate it ⚡ CPU: a few minutes, up to about 7 min on the course JupyterHub**

In [24]:
# TODO: Predict the entities of all test sentences with GLiNER, convert them to IOB tags
# and compute the entity-level micro and macro F1 scores with seqeval, as in Part 1
from seqeval.metrics import classification_report, f1_score
from tqdm.auto import tqdm

# Defined again, so that Part 2 also runs in a new runtime without the training cells of Part 1
id2tag = {i: tag for tag, i in ner_tags.items()}

gliner_true_tags = [[id2tag[tag_id] for tag_id in example["ner_tags"]] for example in test_set]
gliner_pred_tags = []
for example in tqdm(test_set):
    words = example["tokens"]
    entities = gliner_model.predict_entities(" ".join(words), list(gliner_labels), threshold=0.5)
    gliner_pred_tags.append(spans_to_iob(words, entities))

gliner_metrics = {
    "micro_f1": float(f1_score(gliner_true_tags, gliner_pred_tags, average="micro")),
    "macro_f1": float(f1_score(gliner_true_tags, gliner_pred_tags, average="macro")),
}
print("GLiNER Metrics:", gliner_metrics)
print(classification_report(gliner_true_tags, gliner_pred_tags, digits=4))

  0%|          | 0/2000 [00:00<?, ?it/s]

GLiNER Metrics: {'micro_f1': 0.6931694976508854, 'macro_f1': 0.689986655100097}
              precision    recall  f1-score   support

         LOC     0.6759    0.6280    0.6510       973
         ORG     0.6228    0.6561    0.6391       823
         PER     0.7454    0.8177    0.7799       938

   micro avg     0.6850    0.7015    0.6932      2734
   macro avg     0.6814    0.7006    0.6900      2734
weighted avg     0.6838    0.7015    0.6916      2734



In [25]:
# Sentences where the tags predicted by GLiNER differ from the gold tags
disagreements = [i for i in range(len(test_set)) if gliner_true_tags[i] != gliner_pred_tags[i]]
print(f"GLiNER differs from the gold tags in {len(disagreements)} of {len(test_set)} sentences\n")

for i in disagreements[:15]:
    print("Words:", test_set[i]["tokens"])
    print("Gold: ", gliner_true_tags[i])
    print("Pred: ", gliner_pred_tags[i])
    print()

GLiNER differs from the gold tags in 885 of 2000 sentences

Words: ['WEITERLEITUNG', 'Hu', '(', 'Xi’an', ')']
Gold:  ['O', 'B-LOC', 'I-LOC', 'I-LOC', 'I-LOC']
Pred:  ['O', 'B-PER', 'I-PER', 'I-PER', 'I-PER']

Words: ['Runde', 'rammte', 'Lorenzo', 'Bandini', 'an', 'zweiter', 'Stelle', 'liegend', 'die', 'Streckenbegrenzung', 'aus', 'Strohballen', '.']
Gold:  ['O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Pred:  ['B-LOC', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'O']

Words: ["'", "''", 'Gary', 'Glitter', "''", "'", '-', "''Always", 'Yours', "''"]
Gold:  ['O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'O']
Pred:  ['O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'O']

Words: ['Alexandrasittiche', 'kommen', 'außerdem', 'in', 'den', 'Galeriewäldern', 'entlang', 'von', 'Wasserläufen', 'vor', '.']
Gold:  ['O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O']
Pred:  ['B-ORG', 'O', 'O', 'O', 'O', 'B-LOC', 'O', '

📝❓ Compare GLiNER with your four BERT variants. Which of them outperform it, and what does this suggest about how much annotated data is needed? When would you rather use GLiNER, and when a fine-tuned model?

📝❓ Look at some sentences where the tags predicted by GLiNER differ from the gold tags. Is it always GLiNER that is wrong? Keep in mind that WikiANN was annotated automatically from the links in Wikipedia articles.

## Anonymise entities

In [26]:
# TODO: Implement anonymization function
def anonymize_entities(text: str, entities: list[dict]) -> str:
    """Anonymize entities in text by replacing them with tags like <PERSON>, <IBAN> etc.

    Args:
        text (str): The original text
        entities (list): List of entity dictionaries with 'start', 'end', and 'label' keys

    Returns:
        str: Text with entities replaced by the corresponding tags.
    """
    # Replace from the end of the text, so that the offsets of the remaining entities stay valid
    for entity in sorted(entities, key=lambda e: e["start"], reverse=True):
        tag = "<" + entity["label"].upper().replace(" ", "_") + ">"
        text = text[:entity["start"]] + tag + text[entity["end"]:]
    return text

## Your own dataset

Time to create our own, diverse dataset. Make sure to not make it easy for the model (for example, do not mention the label in the sentence, but make sure it can be inferred from the context).

In [27]:
# TODO: Fill these 2 lists with your own data. pii_texts should be of length (at least) 10.
# All people, numbers and addresses below are made up.

pii_texts = [
    "I ordered the package to be delivered at Andreasstrasse 15, 8050 Zürich, and I added a small sticker with \"Lena Hofer\" written on it.",
    "Bitte überweise die Miete für November auf CH93 0076 2011 6238 5295 7, das Konto läuft noch auf den Namen von Marco Bianchi.",
    "Falls du den Schlüssel vergisst, ruf mich einfach unter +41 79 555 01 23 an, ich bin bis 18 Uhr in der Migros am Limmatplatz.",
    "Der Blutdruck von Frau Yvonne Keller, geboren am 14.03.1962, bleibt hoch, deshalb verschreibt Dr. Amrein ihr weiterhin Ramipril gegen ihre Hypertonie.",
    "At the border, the officer scanned X4K7291P8 and wished Tomasz Nowak a pleasant stay before stamping the small burgundy booklet.",
    "The payment with 4539 1488 0343 6467, valid through 09/28, was declined twice at the hotel in Lisbon.",
    "Le colis de Camille Durand est reparti à l'expéditeur : personne n'a répondu au 12 rue des Lilas, 69003 Lyon, ni à camille.durand@exemple.fr.",
    "The server logs show repeated failed logins for jdoe_1987 from 192.168.14.27 shortly after midnight, so Acme Logistics locked the account.",
    "Der weisse Golf mit ZH 482 915 stand wieder in unserer Einfahrt, also habe ich Herrn Peter Brunner von nebenan eine Nachricht geschrieben.",
    "Sofia Rossi lavora da tre anni presso Banca Lombarda a Milano e ha compiuto 34 anni il 2 maggio 1992; scrivile pure a s.rossi@bancalombarda.example.",
    "Auf meiner Lohnabrechnung steht die 756.1234.5678.97, und das Geld kommt jeden Monat von der Kaffeerösterei Bohnenglück GmbH in Bern.",
    "After the interview, send feedback about Priya Raman to hr@northwind.example before Friday; she lands in Geneva at 10:40 on LX 318.",
]

labels = [
    "person", "full address", "location", "organization", "email", "phone number", "IBAN",
    "credit card number", "passport number", "Social Security Number", "date of birth",
    "license plate", "IP address", "username", "medical condition", "medication", "flight number",
]

In [28]:
# TODO: Create dataset
dataset: list[dict] = []
for text in pii_texts:
    entities = gliner_model.predict_entities(text, labels, threshold=0.5)
    dataset.append({
        "text": text,
        "entities": entities,
        "anonymized_text": anonymize_entities(text, entities),
    })

Time to see the results!

In [29]:
for example in dataset:
    print("Original Text:\n", example["text"])
    print("Anonymized Text:\n", example["anonymized_text"])
    print("\n---\n")

Original Text:
 I ordered the package to be delivered at Andreasstrasse 15, 8050 Zürich, and I added a small sticker with "Lena Hofer" written on it.
Anonymized Text:
 I ordered the package to be delivered at <FULL_ADDRESS>, and I added a small sticker with "<PERSON>" written on it.

---

Original Text:
 Bitte überweise die Miete für November auf CH93 0076 2011 6238 5295 7, das Konto läuft noch auf den Namen von Marco Bianchi.
Anonymized Text:
 Bitte überweise die Miete für November auf <PHONE_NUMBER>, das Konto läuft noch auf den Namen von <PERSON>.

---

Original Text:
 Falls du den Schlüssel vergisst, ruf mich einfach unter +41 79 555 01 23 an, ich bin bis 18 Uhr in der Migros am Limmatplatz.
Anonymized Text:
 Falls du den Schlüssel vergisst, ruf mich einfach unter <PHONE_NUMBER> an, ich bin bis 18 Uhr in der <ORGANIZATION>.

---

Original Text:
 Der Blutdruck von Frau Yvonne Keller, geboren am 14.03.1962, bleibt hoch, deshalb verschreibt Dr. Amrein ihr weiterhin Ramipril gegen ihre

## Part 2: Questions

📝❓Discuss the benefits of using GLiNER vs a more traditional NER solution.

📝❓Did you encounter false positives/negatives? Discuss.

---

## Lab report

📝❓ Write your lab report here, addressing all questions in the notebook. For convenience, the questions are collected below; please keep your answers clearly marked.

Our approach: German WikiANN (1,000 / 3,000 training and 2,000 test sentences) with `google-bert/bert-base-german-cased`. Labels go on the first subword of each word; four variants (whole model vs frozen backbone × 1k vs 3k) were trained with the recommended hyperparameters and evaluated with entity-level F1 (`seqeval`). In Part 2, GLiNER (`urchade/gliner_multi_pii-v1`) was run zero-shot on the same test set and used to anonymise 12 own texts.

What worked and results: Full fine-tuning worked best (micro F1 0.8246 with 3,000 sentences); freezing the backbone worked poorly (≤ 0.6574). GLiNER (0.6932) lies in between without any training, and on our texts it masked names reliably but missed some structured identifiers.

### Questions

**Part 1: Data and tokenisation**

- 📝❓ Why do we need separate `B-` and `I-` tags? What could not be represented with only `PER`, `ORG`, `LOC` and `O`?
  - `B-` marks where an entity starts and `I-` its continuation. With only `PER`, `ORG`, `LOC`, `O`, two adjacent entities of the same type look like one: *"Angela Merkel Olaf Scholz"* would be `PER PER PER PER`, while `B-PER I-PER B-PER I-PER` shows the boundary.

- 📝❓ Why do you need to be aware of the longest input length within your dataset? Which parameter of the model dictates this?
  - BERT cannot process inputs longer than its position limit, so longer inputs get truncated and lose words. Knowing the real maximum (`114` subword tokens; average ~`16`) also lets us pad to `114` instead of `512`, which saves memory and compute.
  - The limit is set by `max_position_embeddings` (`512`).

- 📝❓ The dataset is split into words, and the assigned labels are for words. How should we deal with labels after tokenisation?
  - With `word_ids()`: the first subword of each word gets the word's tag; later subwords, special tokens and padding get `-100`, which the loss ignores. So each word counts once and evaluation maps back 1-to-1 to the gold tags.

- 📝❓ What value should replace the question marks in the printed sizes? Should this be the same for all samples? Why/Why not?
  - `114` for `input_ids`, `attention_mask` and `labels`. It is the same for all samples because we pad to `max_length=114`, which is needed to stack them into batches; padding is masked by `attention_mask = 0` and `labels = -100`.


**Part 1: Evaluation and hyperparameters**

- 📝❓ Why do we evaluate NER on the level of entities rather than on the level of individual tokens?
  - `O` is **69.86%** of the test words, so predicting `O` everywhere would score ~0.70 on the token level while finding no entity. `seqeval` ignores `O` and counts an entity as correct only if its boundaries and type both match.

- 📝❓ Why do the variants with a frozen backbone need a larger learning rate?
  - Variants 1–2 adjust a pretrained backbone, so small steps (`5e-5`) avoid destroying what it learned. Variants 3–4 train only the classifier head (`5,383` parameters), which the load report shows as `MISSING`, i.e. randomly initialised. Learning it from scratch needs larger steps (`1e-3`) and more epochs, and the frozen backbone cannot be damaged.

- 📝❓ Is there a challenge when evaluating the predictions of your model? Why is this challenge present and how do you plan to deal with it?
  - The model predicts one tag per subword position (`114`, incl. special tokens and padding), while gold tags are per word, with a different number of words per sentence. We drop all positions labelled `-100` from both sequences in `compute_metrics`, leaving exactly one tag per word.


**Part 1: Results**

- 📝❓ Summarise the four model variants: entity-level micro and macro F1, and the number of trainable parameters.
  | Variant | Entity micro F1 | Entity macro F1 | Trainable parameters |
  |---|---|---|---|
  | 1. Whole model, 1000 sentences | 0.7644 | 0.7607 | 108,496,135 (100%) |
  | 2. Whole model, 3000 sentences | 0.8246 | 0.8210 | 108,496,135 (100%) |
  | 3. Frozen backbone, 1000 sentences | 0.6226 | 0.6160 | 5,383 (0.005%) |
  | 4. Frozen backbone, 3000 sentences | 0.6574 | 0.6524 | 5,383 (0.005%) |

- 📝❓ Compute the token-level micro and macro F1 (including `O`) for one of your variants and compare them with the entity-level micro F1. Why do these scores differ so much?
  - Variant 2: token level (incl. `O`) micro **0.9527** / macro 0.8805 vs entity level micro **0.8246** / macro 0.8210.
  - The gap comes from (1) the many easy `O` tokens, which count on the token level but not on the entity level, and (2) partial credit: `B-ORG I-ORG O` for a 3-word gold `ORG` is 2/3 correct on the token level but a complete miss on the entity level.

- 📝❓ Is it better to freeze or not to freeze the transformer backbone weights? Hypothesise why.
  - Not freezing is clearly better: +0.14 micro F1 at 1,000 sentences and +0.17 at 3,000. Full fine-tuning on 1,000 sentences even beats the frozen backbone on 3,000.
  - BERT was pretrained to predict masked words, not entity boundaries or types. A frozen backbone leaves only one linear layer to do the task, while fine-tuning lets all layers adapt. Freezing only pays off when compute is very limited.


**Part 2: GLiNER**

- 📝❓ Compare GLiNER with your four BERT variants. Which of them outperform it, and what does this suggest about how much annotated data is needed? When would you rather use GLiNER, and when a fine-tuned model?
  - GLiNER (no training): micro **0.6932** / macro 0.6900. Only the two whole-model variants (0.7644, 0.8246) beat it; both frozen variants (0.6226, 0.6574) are below.
  - Little annotated data is needed if the backbone is trained: 1,000 sentences already beat GLiNER by 0.07. Training the backbone matters more than more data (+0.17 vs +0.06 from tripling the data).
  - Use GLiNER when there is no labelled data or the entity types are new; use a fine-tuned model when labelled data for fixed types exists and accuracy matters most.

- 📝❓ Look at some sentences where the tags predicted by GLiNER differ from the gold tags. Is it always GLiNER that is wrong?
  - No. GLiNER differs from gold in 885 of 2,000 sentences, but the automatic WikiANN labels are sometimes wrong: `Gänseküken` (goslings) is gold `LOC`, and `Moor Park`, which a railway line reaches, is gold `ORG` where GLiNER's `LOC` fits better.
  - GLiNER's own typical errors: pronouns and common nouns as entities (`Er` → `PER`, `Geschäftsführung` → `ORG`), wrong boundaries (`Ernst Stromer von Reichenbach` split into `PER` + `LOC`) and wrong types (`Tour d’Aï` → `ORG` instead of `LOC`).

- 📝❓ Discuss the benefits of using GLiNER vs a more traditional NER solution.
  - **Benefits:** no labelled data or training needed; entity types are free-text labels, so new types (e.g. `IBAN`) need no retraining; one model with English labels worked on our English, German, French and Italian texts; it returns character spans with confidence scores, which map directly onto the text and allow a tunable threshold.
  - **Drawbacks:** lower accuracy than a fine-tuned model, and it missed some structured identifiers (e.g. an IP address).

- 📝❓ Did you encounter false positives/negatives on your own dataset? Discuss.
  - Yes. On our 12 texts (threshold 0.5), all names and emails were masked, but:
    - **False negatives:** passport number `X4K7291P8`, IP `192.168.14.27` and licence plate `ZH 482 915` stayed visible, although these labels were in the list.
    - **False positives:** generic words `border` (`<LOCATION>`), `officer` (`<PERSON>`), `hotel` (`<ORGANIZATION>`), and the car `Der weisse Golf` (`<LOCATION>`).
    - **Wrong label/boundary:** the IBAN became `<PHONE_NUMBER>` (still hidden); `Migros am Limmatplatz` became one `<ORGANIZATION>`.
  - For anonymisation, false negatives are worse, since they leave personal data readable. A lower threshold, or one per entity type, would reduce them.


## Declarations

Please fill in both declarations below by editing this cell (double-click it). Mark the applicable option by replacing ☐ with ☒ (copy the symbol from here: ☒) and complete the text field where required. Your reviewers will check that both declarations are present and completed.

### Contribution declaration

By submitting this work, the group declares that the contributions of the group members were:

☒ approximately equal

☐ not approximately equal

If the contributions were not approximately equal, no individual group member needs to be identified.

### Declaration on the use of AI

The group declares that any use of AI-based tools in preparing this work complied with the rules specified for this exercise.

☐ No AI-based tools were used.

☒ AI-based tools were used in accordance with the rules of the exercise.

If AI-based tools were used, briefly state the tool(s) used and the purpose(s) for which they were used:

Claude Opus 5.5 is used for debugging and simplifying code, for graphs, shaping markdown texts (tables, colors etc.), and for understanding certain results. 
